In [ ]:
# ============================================================
# ABLATION STUDY (best fold) 
# A0: Full Proposed
# A1: w/o FEM
# A2: w/o CoordAttn
# A4: w/o BiFPN
# A5: Backbone only
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import timm
import warnings
warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
ABLATION_CFG = {
    "backbone": "mobilenetv3_small_100",
    "batch_size": 48,
    "epochs": 40,
    "lr": 3e-4,
    "warmup_epochs": 3,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "weight_decay": 1e-4,
    "fold_id": 0,                    # Fixed Fold 0
}

# Ablation variants (NO slash in names)
ABLATIONS = {
    "A0_Full_Proposed":  dict(use_fem=True,  use_coord=True,  use_eca=True,  use_bifpn=True),
    "A1_wo_FEM":         dict(use_fem=False, use_coord=True,  use_eca=True,  use_bifpn=True),
    "A2_wo_CoordAttn":   dict(use_fem=True,  use_coord=False, use_eca=True,  use_bifpn=True),
    "A4_wo_BiFPN":       dict(use_fem=True,  use_coord=True,  use_eca=True,  use_bifpn=False),
    "A5_Backbone_only":  dict(use_fem=False, use_coord=False, use_eca=False, use_bifpn=False),
}

# -------------------- MODEL COMPONENTS --------------------
class ECA(nn.Module):
    def __init__(self, c, k=3):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)
        y = self.conv(y).transpose(1, 2).unsqueeze(-1)
        return x * torch.sigmoid(y)

class CoordAttn(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(c, h, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(h)
        self.act = nn.ReLU(inplace=True)
        self.conv_h = nn.Conv2d(h, c, 1, bias=False)
        self.conv_w = nn.Conv2d(h, c, 1, bias=False)
    def forward(self, x):
        n, c, h, w = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)
        return x * torch.sigmoid(self.conv_h(y_h)) * torch.sigmoid(self.conv_w(y_w))

class FrequencyEnhancement(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.high = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 1, bias=False)
        )
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1), nn.Sigmoid())
    def forward(self, x):
        return x + self.high(x) * self.gate(x)

class BiFPNFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.w = nn.Parameter(torch.ones(len(channels)))
        self.eps = 1e-4
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, channels[0], 1, bias=False),
                nn.BatchNorm2d(channels[0]),
                nn.ReLU(inplace=True)
            ) for c in channels
        ])
    def forward(self, feats):
        target = feats[0].shape[-2:]
        resized = []
        for i, f in enumerate(feats):
            f = self.convs[i](f)
            if f.shape[-2:] != target:
                f = F.interpolate(f, size=target, mode="bilinear", align_corners=False)
            resized.append(f)
        w = F.relu(self.w)
        w = w / (w.sum() + self.eps)
        return sum(wi * fi for wi, fi in zip(w, resized))

class DentalNet(nn.Module):
    def __init__(self, backbone="mobilenetv3_small_100", num_classes=4,
                 use_coord=True, use_eca=True, use_fem=True, use_bifpn=True,
                 dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        chs = self.backbone.feature_info.channels()[-3:]

        self.fems = nn.ModuleList([FrequencyEnhancement(c) if use_fem else nn.Identity() for c in chs])
        self.attns = nn.ModuleList()
        for c in chs:
            layers = []
            if use_coord: layers.append(CoordAttn(c))
            if use_eca:   layers.append(ECA(c))
            self.attns.append(nn.Sequential(*layers) if layers else nn.Identity())

        self.use_bifpn = use_bifpn
        if use_bifpn:
            self.fusion = BiFPNFusion(chs)
            fuse_ch = chs[0]
        else:
            fuse_ch = sum(chs)

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(fuse_ch * 2, num_classes)

    def forward_feats(self, x):
        feats = self.backbone(x)[-3:]
        feats = [fem(f) for fem, f in zip(self.fems, feats)]
        feats = [attn(f) for attn, f in zip(self.attns, feats)]
        return feats

    def classify(self, feats):
        if self.use_bifpn:
            fused = self.fusion(feats)
            pooled = torch.cat([
                F.adaptive_avg_pool2d(fused, 1).flatten(1),
                F.adaptive_max_pool2d(fused, 1).flatten(1)
            ], dim=1)
        else:
            pooled = torch.cat([
                torch.cat([
                    F.adaptive_avg_pool2d(f, 1).flatten(1),
                    F.adaptive_max_pool2d(f, 1).flatten(1)
                ], dim=1) for f in feats
            ], dim=1)
        return self.fc(self.drop(pooled))

    def forward(self, x):
        return self.classify(self.forward_feats(x))

def build_ablation_model(cfg_dict, pretrained=True):
    return DentalNet(
        backbone=ABLATION_CFG["backbone"],
        num_classes=len(CLASSES),
        use_coord=cfg_dict["use_coord"],
        use_eca=cfg_dict["use_eca"],
        use_fem=cfg_dict["use_fem"],
        use_bifpn=cfg_dict["use_bifpn"],
        dropout=ABLATION_CFG["dropout"],
        pretrained=pretrained
    ).to(device)

# -------------------- DATALOADER (warning free) --------------------
def make_loader(idx, tf, shuffle):
    return DataLoader(
        DentalDS(X, y, idx, tf),
        batch_size=ABLATION_CFG["batch_size"],
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        drop_last=shuffle
    )

# -------------------- TRAIN / EVAL --------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(xb)
        P.append(out.float().softmax(1).cpu())
        Y.append(yb)
    P = torch.cat(P).numpy()
    Y = torch.cat(Y).numpy()
    return P, Y, P.argmax(1)

def train_ablation(name, cfg_dict, fold_id=0):
    print(f"\n{'='*70}")
    print(f"🔧 Ablation: {name}")
    print(f"   Config → FEM={cfg_dict['use_fem']} | Coord={cfg_dict['use_coord']} | "
          f"ECA={cfg_dict['use_eca']} | BiFPN={cfg_dict['use_bifpn']}")
    print(f"{'='*70}")

    fd = folds[fold_id]
    tr_loader = make_loader(fd["train"], train_tf, True)
    va_loader = make_loader(fd["val"],   eval_tf, False)
    te_loader = make_loader(fd["test"],  eval_tf, False)

    cnt = np.bincount(y[fd["train"]], minlength=len(CLASSES))
    w = torch.tensor(len(fd["train"]) / (len(CLASSES) * cnt + 1e-6), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=ABLATION_CFG["label_smoothing"])

    model = build_ablation_model(cfg_dict)
    params = sum(p.numel() for p in model.parameters()) / 1e6

    head_params = (list(model.fems.parameters()) + list(model.attns.parameters()) +
                   list(model.fc.parameters()))
    if cfg_dict["use_bifpn"]:
        head_params += list(model.fusion.parameters())

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": ABLATION_CFG["lr"]},
        {"params": head_params, "lr": ABLATION_CFG["lr"] * 3},
    ], weight_decay=ABLATION_CFG["weight_decay"])

    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=ABLATION_CFG["warmup_epochs"]),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(ABLATION_CFG["epochs"] - ABLATION_CFG["warmup_epochs"], 1))
    ], milestones=[ABLATION_CFG["warmup_epochs"]])

    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_f1 = -1.0
    ckpt = os.path.join(OUT_DIR, f"ablation_{name}.pth")
    hist = []

    for ep in range(ABLATION_CFG["epochs"]):
        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in tqdm(tr_loader, desc=f"{name} | Ep {ep+1}", leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * len(yb)
            n += len(yb)
        sched.step()

        # Validation
        _, labels, preds = evaluate(model, va_loader)
        val_acc = accuracy_score(labels, preds)
        val_f1  = f1_score(labels, preds, average="macro")
        hist.append({"epoch": ep+1, "train_loss": run_loss/n, "val_acc": val_acc, "val_f1": val_f1})

        flag = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), ckpt)
            flag = " ✅"
        print(f"Ep {ep+1:02d} | loss {run_loss/n:.4f} | val_acc {val_acc:.4f} | macroF1 {val_f1:.4f}{flag}")

    # Load best & Test
    model.load_state_dict(torch.load(ckpt, map_location=device))
    probs, labels, preds = evaluate(model, te_loader)

    acc = accuracy_score(labels, preds) * 100
    macro_f1 = f1_score(labels, preds, average="macro") * 100
    weighted_f1 = f1_score(labels, preds, average="weighted") * 100

    # Caries TPR
    cm = confusion_matrix(labels, preds)
    caries_idx = CLASSES.index("Caries")
    caries_tpr = cm[caries_idx, caries_idx] / (cm[caries_idx].sum() + 1e-8) * 100

    result = {
        "Variant": name,
        "Accuracy (%)": round(acc, 2),
        "Macro-F1 (%)": round(macro_f1, 2),
        "Weighted-F1 (%)": round(weighted_f1, 2),
        "Caries TPR (%)": round(caries_tpr, 2),
        "Params (M)": round(params, 2),
    }
    print(f"\n✅ {name} → Acc: {acc:.2f}% | Macro-F1: {macro_f1:.2f}% | Caries TPR: {caries_tpr:.2f}% | Params: {params:.2f}M")
    return result, pd.DataFrame(hist), model

# ============================================================
# RUN ALL ABLATIONS
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)

ablation_results = []
ablation_hists = {}

for name, cfg in ABLATIONS.items():
    res, hist, _ = train_ablation(name, cfg, fold_id=ABLATION_CFG["fold_id"])
    ablation_results.append(res)
    ablation_hists[name] = hist
    hist.to_csv(os.path.join(OUT_DIR, f"ablation_hist_{name}.csv"), index=False)

# -------------------- FINAL ABLATION TABLE --------------------
df = pd.DataFrame(ablation_results)

# Delta Macro-F1 from Full model
full_f1 = df.loc[df["Variant"] == "A0_Full_Proposed", "Macro-F1 (%)"].values[0]
df["Δ Macro-F1"] = (df["Macro-F1 (%)"] - full_f1).round(2)

print("\n" + "="*95)
print("ABLATION STUDY RESULTS (Fold 0)")
print("="*95)
print(df.to_string(index=False))
print("="*95)

df.to_csv(os.path.join(OUT_DIR, "ablation_table.csv"), index=False)

# -------------------- PLOT --------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Macro-F1 comparison
ax = axes[0]
colors = ["#2ecc71" if "Full" in v else "#3498db" for v in df["Variant"]]
bars = ax.barh(df["Variant"], df["Macro-F1 (%)"], color=colors, edgecolor="black")
ax.set_xlabel("Macro-F1 (%)")
ax.set_title("Macro-F1 Comparison")
ax.set_xlim(0, 105)
for bar, val in zip(bars, df["Macro-F1 (%)"]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va="center", fontsize=10)
ax.grid(True, axis="x", alpha=0.3)

# 2. Caries TPR
ax = axes[1]
bars = ax.barh(df["Variant"], df["Caries TPR (%)"], color="#e74c3c", edgecolor="black")
ax.set_xlabel("Caries TPR (%)")
ax.set_title("Caries Recall (TPR)")
ax.set_xlim(0, 105)
for bar, val in zip(bars, df["Caries TPR (%)"]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va="center", fontsize=10)
ax.grid(True, axis="x", alpha=0.3)

# 3. Parameters
ax = axes[2]
bars = ax.barh(df["Variant"], df["Params (M)"], color="#9b59b6", edgecolor="black")
ax.set_xlabel("Parameters (M)")
ax.set_title("Model Size")
for bar, val in zip(bars, df["Params (M)"]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f"{val:.2f}M", va="center", fontsize=10)
ax.grid(True, axis="x", alpha=0.3)

plt.suptitle("Ablation Study | DentAttn-MSFNet (Fold 0)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ablation_study.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Ablation completed! Results saved to: {OUT_DIR}")
print("   - ablation_table.csv")
print("   - ablation_study.png")
print("   - ablation_hist_*.csv")
print("   - ablation_*.pth")